# Phantasm — Unsloth Fine-Tuning Pipeline (Llama 3.1 8B)

Fine-tuning on Discord DM persona dataset. Runs on Google Colab (Free T4 or better).
Outputs quantized GGUF for local inference.

In [ ]:
# 1. Install Unsloth & dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Load model and tokenizer
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None  # None for auto detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

In [ ]:
# 3. Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# 4. Upload & format dataset
# Upload phantasm_train_sharegpt.jsonl to Colab files
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

dataset = load_dataset("json", data_files="phantasm_train_sharegpt.jsonl", split="train")
dataset = standardize_sharegpt(dataset)


def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}


dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# 5. Setup Trainer & Train
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=120,  # adjust as needed (e.g. 100-300)
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 6. Test Model Inference
FastLanguageModel.for_inference(model)
messages = [
    {"from": "human", "value": "yo what are you doing today?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True)
print(tokenizer.batch_decode(outputs[:, inputs.shape[1] :], skip_special_tokens=True)[0])

In [ ]:
# 7. Export to GGUF for local execution (q4_k_m recommended for CPU inference)
model.save_pretrained_gguf("phantasm_model", tokenizer, quantization_method="q4_k_m")
# Download the generated .gguf file to your machine